In [ ]:
# In[1]:

import os
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# helper to load and summarize files
def summarize_table(path, key_name=None, sample_messages=False):
    if not os.path.exists(path):
        return {"missing_or_empty": True, "note": "file not found"}
    try:
        df = pd.read_csv(path)
    except Exception as e:
        return {"missing_or_empty": True, "note": f"failed to read CSV: {e}"}
    if df is None or df.shape[0] == 0:
        return {"missing_or_empty": True, "note": "file is empty"}
    # ensure timestamp column exists
    if 'timestamp' not in df.columns:
        return {"missing_or_empty": True, "note": "missing 'timestamp' column"}
    # Ensure timestamp integers (do not convert timezone; timestamps remain ints as requested)
    ts_series = df['timestamp'].dropna().astype(int)
    row_count = int(df.shape[0])
    min_ts = int(ts_series.min()) if not ts_series.empty else None
    max_ts = int(ts_series.max()) if not ts_series.empty else None
    # unique cmdb ids
    if 'cmdb_id' in df.columns:
        unique_cmdb = sorted(df['cmdb_id'].dropna().unique())[:50]
    else:
        unique_cmdb = []
    result = {
        "row_count": row_count,
        "min_timestamp": min_ts,
        "max_timestamp": max_ts,
        "unique_cmdb_ids": list(unique_cmdb)
    }
    # additional handling for message samples (error_logs)
    if sample_messages:
        if 'message' not in df.columns:
            result["sample_messages"] = []
            return result
        # get up to 5 distinct messages with their earliest timestamp and corresponding cmdb_id
        # group by message and pick row with earliest timestamp
        grp = df.dropna(subset=['message']).copy()
        if grp.shape[0] == 0:
            result["sample_messages"] = []
            return result
        grp['timestamp'] = grp['timestamp'].astype(int)
        idx = grp.groupby('message')['timestamp'].idxmin()
        sample_rows = grp.loc[idx].sort_values('timestamp').head(5)
        samples = []
        for _, r in sample_rows.iterrows():
            samples.append({
                "message": str(r['message']),
                "timestamp": int(r['timestamp']),
                "cmdb_id": r['cmdb_id'] if 'cmdb_id' in r and pd.notna(r['cmdb_id']) else None
            })
        result["sample_messages"] = samples
        return result
    # for metric/trace/log, compute top name counts
    if key_name:
        if key_name not in df.columns:
            result[f"top_{key_name}s"] = {}
            return result
        name_counts = df[key_name].dropna()
        top_counts = name_counts.value_counts().head(20)
        # convert to dict of name->int
        result_key = f"top_{key_name}s"
        result[result_key] = top_counts.astype(int).to_dict()
    return result

metric_summary = summarize_table('metric.csv', key_name='kpi_name')
trace_summary = summarize_table('trace.csv', key_name='trace_name')
log_summary = summarize_table('log.csv', key_name='log_name')
error_logs_summary = summarize_table('error_logs.csv', sample_messages=True)

summary = {
    "metric.csv": metric_summary,
    "trace.csv": trace_summary,
    "log.csv": log_summary,
    "error_logs.csv": error_logs_summary
}

summary

```
Out[1]:
```
summary_text = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv: 319,507 rows. Timestamp range: 1647734400 to 1647738000. "
    "About 54 unique cmdb_ids (services, pods, and node-1..node-6); examples: adservice, frontend-0, paymentservice-1, node-6. "
    "Top KPI names are dominated by node-6 container metrics (e.g., network and fs metrics), with the highest-count KPIs including container.node-6.container_network_transmit_packets.eth0 and several other node-6 metrics.\n\n"
    "trace.csv: 31,120 rows. Timestamp range: 1647736200 to 1647737940. "
    "Unique cmdb_ids are mainly service pods (frontend-0..2, paymentservice-*, shippingservice-*, etc.). "
    "Top trace names are mainly traces from frontend pods (trace.from_frontend-1/2/0.*) and trace.self.*, indicating frontend-origin traces are the most frequent.\n\n"
    "log.csv: 1,808 rows. Timestamp range: 1647736200 to 1647738000. "
    "Unique cmdb_ids include various service pods and redis-cart-0. Top log names are evenly split between 'log.error_count' and 'log.row_count' (904 occurrences each).\n\n"
    "error_logs.csv: 199 rows. Timestamp range: 1647737332 to 1647737513. "
    "All error log entries come from frontend pods (frontend-0, frontend-1, frontend-2). "
    "Sample message (earliest distinct): 'severity: warning, message: failed to retrieve ads' at timestamp 1647737332 on frontend-1.\n\n"
    "Overall impression: Metrics are heavily populated with node-6 container metrics; traces are dominated by frontend-related traces; logs show error counts evenly present; error messages are concentrated on frontend pods with a repeated warning about failing to retrieve ads."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'row_count': 319507, 'min_timestamp': 1647734400, 'max_timestamp': 1647738000, 'unique_cmdb_ids': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1'], 'top_kpi_names': {'container.node-6.container_network_transmit_packets.eth0': 2196, 'container.node-6.container_network_receive_packets_dropped.eth0': 2196, 'container.node-6.container_network_receive_MB.eth0': 2196, 'container.node-6.container_network_receive_errors.eth0': 2196, 'container.node-6.container_network_receive_packets.eth0': 2196, 'container.node-6.container_network_transmit_errors.eth0': 2196, 'container.node-6.container_network_transmit_MB.eth0': 2196, 'container.node-6.container_network_transmit_packets_dropped.eth0': 2196, 'container.node-6.container_memory_failures.hierarchy.pgfault': 2074, 'container.node-6.container_spec_cpu_quota': 2074, 'container.node-6.container_memory_failures.container.pgmajfault': 2074, 'container.node-6.container_fs_read_seconds./dev/vda1': 2074, 'container.node-6.container_fs_limit_MB./dev/vda1': 2074, 'container.node-6.container_fs_reads./dev/vda1': 2074, 'container.node-6.container_fs_reads./dev/vda': 2074, 'container.node-6.container_spec_cpu_period': 2074, 'container.node-6.container_spec_memory_swap_limit_MB': 2074, 'container.node-6.container_cpu_load_average_10s': 2074, 'container.node-6.container_cpu_system_seconds': 2074, 'container.node-6.container_fs_writes./dev/vda': 2074}}, 'trace.csv': {'row_count': 31120, 'min_timestamp': 1647736200, 'max_timestamp': 1647737940, 'unique_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0'], 'top_trace_names': {'trace.from_frontend-1.row_count': 585, 'trace.from_frontend-1.error_rate': 585, 'trace.from_frontend-1.duration_p95': 585, 'trace.from_frontend-1.duration_mean': 585, 'trace.from_frontend-2.row_count': 584, 'trace.from_frontend-2.error_rate': 584, 'trace.from_frontend-2.duration_p95': 584, 'trace.from_frontend-2.duration_mean': 584, 'trace.from_frontend-0.duration_mean': 579, 'trace.from_frontend-0.duration_p95': 579, 'trace.from_frontend-0.error_rate': 579, 'trace.from_frontend-0.row_count': 579, 'trace.self.row_count': 463, 'trace.self.error_rate': 463, 'trace.self.duration_p95': 463, 'trace.self.duration_mean': 463, 'trace.from_checkoutservice-2.duration_mean': 396, 'trace.from_checkoutservice-2.duration_p95': 396, 'trace.from_checkoutservice-2.error_rate': 396, 'trace.from_checkoutservice-2.row_count': 396}}, 'log.csv': {'row_count': 1808, 'min_timestamp': 1647736200, 'max_timestamp': 1647738000, 'unique_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'redis-cart-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2'], 'top_log_names': {'log.error_count': 904, 'log.row_count': 904}}, 'error_logs.csv': {'row_count': 199, 'min_timestamp': 1647737332, 'max_timestamp': 1647737513, 'unique_cmdb_ids': ['frontend-0', 'frontend-1', 'frontend-2'], 'sample_messages': [{'message': 'severity: warning, message: failed to retrieve ads', 'timestamp': 1647737332, 'cmdb_id': 'frontend-1'}]}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# reuse existing df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# prepare columns
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using entire series
group_keys = ['cmdb_id', 'kpi_name']
global_p95 = (
    df_metric
    .dropna(subset=['value'])
    .groupby(group_keys)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)

# 2) Define incident window in UTC+8 (Asia/Shanghai): 2022-03-20 08:30:00 to 09:00:00
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 20, 8, 30, 0))
end_dt = tz.localize(datetime(2022, 3, 20, 9, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# 3) Filter metric data to the incident window (inclusive)
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# prepare empty result if no points in window
if df_window.shape[0] == 0:
    result_df = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_P95'])
    total_anomalous_pairs = 0
else:
    # 4) Merge global P95 into windowed data
    df_window = df_window.merge(global_p95, on=group_keys, how='left')

    # 5) Mark anomalies: value > global_P95
    df_window['is_anomaly'] = df_window['value'] > df_window['global_P95']

    # 6) Aggregate basic stats per (cmdb_id, kpi_name) for points in window
    agg_basic = (
        df_window
        .groupby(group_keys)
        .agg(
            anomaly_count = ('is_anomaly', 'sum'),
            max_value_in_window = ('value', 'max'),
            points_in_window = ('value', 'count'),
            global_P95 = ('global_P95', 'first')
        )
        .reset_index()
    )

    # 7) Compute earliest anomaly timestamp per group (where is_anomaly is True)
    anomalies = df_window[df_window['is_anomaly']].copy()
    if anomalies.shape[0] > 0:
        earliest = anomalies.groupby(group_keys)['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})
        agg = agg_basic.merge(earliest, on=group_keys, how='left')
    else:
        agg = agg_basic.copy()
        agg['earliest_anomaly_timestamp'] = pd.NA

    # 8) Keep only groups with anomaly_count > 0
    agg = agg[agg['anomaly_count'] > 0].copy()

    # ensure types: earliest timestamp as Int64 (nullable)
    if 'earliest_anomaly_timestamp' in agg.columns:
        agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].astype('Int64')

    # select and order required columns
    result_df = agg[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_value_in_window','global_P95']]

    # sort by anomaly_count desc, then earliest timestamp asc, limit to top 20
    result_df = result_df.sort_values(by=['anomaly_count','earliest_anomaly_timestamp'], ascending=[False, True]).head(20).reset_index(drop=True)

    # total distinct pairs with anomaly_count > 0
    total_anomalous_pairs = int(agg.shape[0])

# Display compact results: top rows and total count
result_df, total_anomalous_pairs

```
Out[2]:
```
I computed the global P95 (95th percentile) for each (cmdb_id, kpi_name) across the entire metric series, then looked at the incident window March 20, 2022 08:30:00–09:00:00 (UTC+8). Key findings:

- Total distinct (cmdb_id, kpi_name) pairs with anomalies (points in the window > global P95): 1,205.
- Top 20 anomalous pairs (limited output) — all have anomaly_count = 3 — sorted by anomaly_count (desc). Earliest anomaly timestamps are either 2022-03-20 08:30:00, 08:31:00, or 08:32:00 (UTC+8). Selected top rows:

  1. cartservice-1, container.node-6.container_network_transmit_MB.eth0
     - anomaly_count: 3
     - earliest_anomaly_timestamp: 2022-03-20 08:30:00 (1647736200)
     - max_value_in_window: 1.377148
     - global_P95: 1.201625

  2. emailservice-0, mesh.destination.checkoutservice.emailservice.... 
     - anomaly_count: 3; earliest: 08:30:00; max: 3027.5; global_P95: 2515.0

  3. frontend-0, mesh.destination.unknown.frontend.istio_request...
     - anomaly_count: 3; earliest: 08:30:00; max: 13912.5; global_P95: 13135.0

  4. frontend-0, mesh.destination.unknown.frontend.istio_response...
     - anomaly_count: 3; earliest: 08:30:00; max: 417325.0; global_P95: 390250.0

  5–14. Additional frontend-0 mesh source/destination metrics (all anomaly_count = 3, earliest 08:30:00), with max values exceeding their respective global P95s.

  15–16. cartservice2-0 network receive/transmit packet metrics
     - anomaly_count: 3; earliest: 08:31:00 (1647736260); max values ~373.33 / 309.67; global P95s 350 / 308

  17. currencyservice-2, container.node-6.container_cpu_cfs_periods
     - anomaly_count: 3; earliest: 08:31:00; max: 165.0; global_P95: 153.5

  18. node-2, system.load.15
     - anomaly_count: 3; earliest: 08:31:00; max: 1.25; global_P95: 1.14

  19. node-5, system.cpu.pct_usage
     - anomaly_count: 3; earliest: 08:32:00 (1647736320); max: 11.89; global_P95: 7.91

- Overall interpretation: During the 08:30–09:00 UTC+8 window, many mesh metrics from frontend pods, several container/network KPIs (notably tied to node-5 and node-6 containers), and some node-level KPIs (load and CPU usage) exceeded their historical 95th percentile. This suggests a concentrated anomaly burst around 08:30–08:32 affecting frontend mesh traffic and certain node/container network and CPU metrics.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window     global_P95
0       cartservice-1  container.node-6.container_network_transmit_MB...              3                  1647736200             1.377148       1.201625
1      emailservice-0  mesh.destination.checkoutservice.emailservice....              3                  1647736200          3027.500000    2515.000000
2          frontend-0  mesh.destination.unknown.frontend.istio_reques...              3                  1647736200         13912.500000   13135.000000
3          frontend-0  mesh.destination.unknown.frontend.istio_respon...              3                  1647736200        417325.000000  390250.000000
4          frontend-0  mesh.source.frontend.cartservice.istio_request...              3                  1647736200            40.500000      39.000000
5          frontend-0  mesh.source.frontend.cartservice.istio_request...              3                  1647736200            40.500000      39.000000
6          frontend-0  mesh.source.frontend.cartservice.istio_respons...              3                  1647736200            40.500000      39.000000
7          frontend-0  mesh.source.frontend.jaeger-collector.istio_re...              3                  1647736200        571300.000000  533325.000000
8          frontend-0  mesh.source.frontend.productcatalogservice.ist...              3                  1647736200           212.500000     204.500000
9          frontend-0  mesh.source.frontend.productcatalogservice.ist...              3                  1647736200           212.500000     204.500000
10         frontend-0  mesh.source.frontend.productcatalogservice.ist...              3                  1647736200         71865.000000   66332.500000
11         frontend-0  mesh.source.frontend.productcatalogservice.ist...              3                  1647736200           212.500000     204.500000
12         frontend-0  mesh.source.frontend.recommendationservice.ist...              3                  1647736200             0.149300       0.136150
13         frontend-0  mesh.source.frontend.recommendationservice.ist...              3                  1647736200            33.000000      32.000000
14         frontend-0  mesh.source.frontend.recommendationservice.ist...              3                  1647736200            33.000000      32.000000
15     cartservice2-0  container.node-5.container_network_receive_pac...              3                  1647736260           373.333333     350.000000
16     cartservice2-0  container.node-5.container_network_transmit_pa...              3                  1647736260           309.666667     308.000000
17  currencyservice-2         container.node-6.container_cpu_cfs_periods              3                  1647736260           165.000000     153.500000
18             node-2                                     system.load.15              3                  1647736260             1.250000       1.140000
19             node-5                               system.cpu.pct_usage              3                  1647736320            11.890000       7.910000, 1205)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse existing variables if present, otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# ensure global_p95 exists (compute if missing)
try:
    global_p95
except NameError:
    group_keys = ['cmdb_id', 'kpi_name']
    global_p95 = (
        df_metric
        .dropna(subset=['value'])
        .groupby(group_keys)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'global_P95'})
    )

# incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022, 3, 20, 8, 30, 0))
end_dt = tz.localize(datetime(2022, 3, 20, 9, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# filter metric to window
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# merge global P95 into windowed data
df_window = df_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# compute ratio and mark anomalies (value > global_P95), avoid division by zero
df_window = df_window.dropna(subset=['value','global_P95']).copy()
df_window['global_P95_float'] = df_window['global_P95'].astype(float)
# handle zero thresholds: if global_P95 == 0, set ratio = inf when value>0
df_window['ratio'] = np.where(df_window['global_P95_float'] == 0,
                              np.where(df_window['value'] > 0, np.inf, 0.0),
                              df_window['value'] / df_window['global_P95_float'])
df_window['is_anomaly'] = df_window['value'] > df_window['global_P95_float']

# restrict to anomalous points
df_anom = df_window[df_window['is_anomaly']].copy()

# define allowed root-cause components (nodes, pods, services from rules)
nodes = [f'node-{i}' for i in range(1,7)]
pods = [
 'frontend-0','frontend-1','frontend-2','frontend2-0',
 'shippingservice-0','shippingservice-1','shippingservice-2','shippingservice2-0',
 'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
 'currencyservice-0','currencyservice-1','currencyservice-2','currencyservice2-0',
 'adservice-0','adservice-1','adservice-2','adservice2-0',
 'emailservice-0','emailservice-1','emailservice-2','emailservice2-0',
 'cartservice-0','cartservice-1','cartservice-2','cartservice2-0',
 'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','productcatalogservice2-0',
 'recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0',
 'paymentservice-0','paymentservice-1','paymentservice-2','paymentservice2-0'
]
services = [
 'frontend','shippingservice','checkoutservice','currencyservice',
 'adservice','emailservice','cartservice','productcatalogservice',
 'recommendationservice','paymentservice'
]
allowed_components = set(nodes + pods + services)

# filter anomalies to allowed components
df_anom = df_anom[df_anom['cmdb_id'].isin(allowed_components)].copy()

# if no anomalies, return empty compact result
if df_anom.shape[0] == 0:
    result_df = pd.DataFrame(columns=['cmdb_id','distinct_kpi_count','earliest_anomaly_timestamp','max_breach_ratio','top_kpis'])
    components_count = 0
else:
    # per (cmdb_id, kpi_name) compute max ratio and earliest anomaly timestamp
    per_kpi = (
        df_anom
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(max_ratio=('ratio','max'),
             earliest_kpi_anom_ts=('timestamp','min'))
    )

    # per component aggregations
    distinct_kpi = per_kpi.groupby('cmdb_id')['kpi_name'].nunique().rename('distinct_kpi_count')
    earliest_comp_ts = per_kpi.groupby('cmdb_id')['earliest_kpi_anom_ts'].min().rename('earliest_anomaly_timestamp')
    max_breach = per_kpi.groupby('cmdb_id')['max_ratio'].max().rename('max_breach_ratio')

    comp_stats = pd.concat([distinct_kpi, earliest_comp_ts, max_breach], axis=1).reset_index()

    # prepare top_kpis (up to 3 per component) with their ratios
    per_kpi_sorted = per_kpi.sort_values(['cmdb_id','max_ratio'], ascending=[True, False]).copy()
    # build top kpis list
    topk_list = (
        per_kpi_sorted
        .groupby('cmdb_id')
        .apply(lambda df: df.head(3)[['kpi_name','max_ratio']].to_dict(orient='records'))
        .rename('top_kpis')
        .reset_index()
    )

    comp_stats = comp_stats.merge(topk_list, on='cmdb_id', how='left')

    # clean and format: round ratios for readability
    comp_stats['max_breach_ratio'] = comp_stats['max_breach_ratio'].replace(np.inf, np.inf)
    # round numeric ratios to 6 decimals, keep inf as-is
    def round_ratio(x):
        if pd.isna(x):
            return x
        if np.isinf(x):
            return np.inf
        return float(np.round(x,6))
    comp_stats['max_breach_ratio'] = comp_stats['max_breach_ratio'].map(round_ratio)
    # also map ratios inside top_kpis
    def format_topk(list_objs):
        if not isinstance(list_objs, list):
            return []
        out = []
        for it in list_objs:
            r = it.get('max_ratio', None)
            out.append({'kpi_name': it.get('kpi_name'), 'ratio': (np.inf if (r is not None and np.isinf(r)) else (None if r is None else float(np.round(r,6))))})
        return out
    comp_stats['top_kpis'] = comp_stats['top_kpis'].map(format_topk)

    # ensure earliest_anomaly_timestamp is Int64
    comp_stats['earliest_anomaly_timestamp'] = comp_stats['earliest_anomaly_timestamp'].astype('Int64')

    # filter only allowed components (already applied) and sort by distinct_kpi_count desc then max_breach_ratio desc
    result_df = comp_stats.sort_values(by=['distinct_kpi_count','max_breach_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

    components_count = int(comp_stats.shape[0])

# Display compact results
result_df, components_count

```
Out[3]:
```
Summary of component-level anomaly aggregation for the incident window (2022-03-20 08:30:00–09:00:00 UTC+8):

Total distinct components (cmdb_id) with at least one KPI breaching its global P95 in the window: 54.

Top 20 components (sorted by distinct_kpi_count desc, then max_breach_ratio desc):

1. frontend-1 — distinct_kpi_count: 73 — earliest_anomaly_timestamp: 1647736200 (2022-03-20 08:30:00 UTC+8) — max_breach_ratio: inf  
   top_kpis: container.node-5.container_fs_reads... (ratio: inf), container.node-5.container_fs_writes... (ratio: inf), container.node-5.container_fs_read_seconds... (ratio: inf)

2. frontend-0 — distinct_kpi_count: 64 — earliest_anomaly_timestamp: 1647736200 (2022-03-20 08:30:00) — max_breach_ratio: inf  
   top_kpis: container.node-6.container_memory_failures... (ratio: inf), mesh.destination.unknown.frontend.istio_response... (ratio: ~1.068), mesh.destination.unknown.frontend.istio_request... (ratio: ~1.057)

3. frontend-2 — distinct_kpi_count: 60 — earliest_anomaly_timestamp: 1647736260 (2022-03-20 08:31:00) — max_breach_ratio: inf  
   top_kpis: container.node-5.container_memory_failures... (ratio: inf), mesh.destination.unknown.frontend... (ratio: >1), ...

4. frontend2-0 — distinct_kpi_count: 57 — earliest_anomaly_timestamp: 1647736200 (08:30:00) — max_breach_ratio: 78.125000  
   top_kpis: mesh.destination.unknown.frontend... (ratio: 78.125), ...

5. adservice — distinct_kpi_count: 42 — earliest_anomaly_timestamp: 1647736320 (08:32:00) — max_breach_ratio: inf  
   top_kpis: runtime.java_lang_Memory_Object... (ratio: inf), container.node-6.container_fs_reads... (ratio: inf), ...

6. productcatalogservice-1 — distinct_kpi_count: 36 — earliest_anomaly_timestamp: 1647736560 (08:36:00) — max_breach_ratio: 1.869276  
   top_kpis: container.node-6.container_network_... (ratio: ~1.869), ...

7. productcatalogservice-2 — distinct_kpi_count: 34 — earliest_anomaly_timestamp: 1647736500 (08:35:00) — max_breach_ratio: inf  
   top_kpis: mesh.destination.recommendation... (ratio: inf), ...

8. productcatalogservice-0 — distinct_kpi_count: 32 — earliest_anomaly_timestamp: 1647736260 (08:31:00) — max_breach_ratio: inf  
   top_kpis: mesh.destination.recommendation... (ratio: inf), ...

9. checkoutservice-0 — distinct_kpi_count: 32 — earliest_anomaly_timestamp: 1647736260 (08:31:00) — max_breach_ratio: 11.619048  
   top_kpis: mesh.source.checkoutservice.jaeger... (ratio: ~11.619), ...

10. node-5 — distinct_kpi_count: 29 — earliest_anomaly_timestamp: 1647736200 (08:30:00) — max_breach_ratio: 5.176471  
    top_kpis: system.io.rkb_s (ratio: ~5.176), system.io.wkb_s (ratio: ...), ...

11. cartservice-0 — distinct_kpi_count: 28 — earliest_anomaly_timestamp: 1647736260 (08:31:00) — max_breach_ratio: inf  
    top_kpis: container.node-6.container_cpu_... (ratio: inf), ...

12. adservice-0 — distinct_kpi_count: 26 — earliest_anomaly_timestamp: 1647736560 (08:36:00) — max_breach_ratio: inf  
    top_kpis: container.node-6.container_fs_reads... (ratio: inf), ...

13. adservice-2 — distinct_kpi_count: 25 — earliest_anomaly_timestamp: 1647736200 (08:30:00) — max_breach_ratio: inf  
    top_kpis: container.node-5.container_fs_reads... (ratio: inf), ...

14. node-4 — distinct_kpi_count: 25 — earliest_anomaly_timestamp: 1647736260 (08:31:00) — max_breach_ratio: inf  
    top_kpis: system.io.avg_q_sz (ratio: inf), ...

15. node-1 — distinct_kpi_count: 25 — earliest_anomaly_timestamp: 1647736200 (08:30:00) — max_breach_ratio: 33.611111  
    top_kpis: system.net.tcp.retrans_segs (ratio: ~33.611), ...

16. productcatalogservice2-0 — distinct_kpi_count: 25 — earliest_anomaly_timestamp: 1647736440 (08:34:00) — max_breach_ratio: 3.539683  
    top_kpis: mesh.source.productcatalogservice... (ratio: ~3.540), ...

17. adservice2-0 — distinct_kpi_count: 24 — earliest_anomaly_timestamp: 1647736200 (08:30:00) — max_breach_ratio: inf  
    top_kpis: container.node-6.container_fs_reads... (ratio: inf), ...

18. recommendationservice-1 — distinct_kpi_count: 24 — earliest_anomaly_timestamp: 1647736260 (08:31:00) — max_breach_ratio: 8.666667  
    top_kpis: mesh.source.recommendationservice... (ratio: ~8.667), ...

19. currencyservice-0 — distinct_kpi_count: 24 — earliest_anomaly_timestamp: 1647736260 (08:31:00) — max_breach_ratio: 1.266667  
    top_kpis: container.node-6.container_memory... (ratio: ~1.267), ...

20. cartservice2-0 — distinct_kpi_count: 23 — earliest_anomaly_timestamp: 1647736260 (08:31:00) — max_breach_ratio: inf  
    top_kpis: container.node-5.container_fs_reads... (ratio: inf), ...

Notes / interpretation:
- Many components (frontend pods, several ad/productcatalog/cart service pods, and multiple nodes) show broad KPI breaches in the window (high distinct_kpi_count).
- Several components show max_breach_ratio = inf, indicating those KPIs had global_P95 == 0 while observed values > 0 during the window (i.e., crossing from zero-baseline to nonzero traffic/usage).
- Earliest anomalies cluster at 08:30–08:32 UTC+8.
- This aggregation highlights frontend pods and several service pods + node resources as the most impacted components during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  distinct_kpi_count  earliest_anomaly_timestamp  max_breach_ratio                                           top_kpis
0                 frontend-1                  73                  1647736200               inf  [{'kpi_name': 'container.node-5.container_fs_r...
1                 frontend-0                  64                  1647736200               inf  [{'kpi_name': 'container.node-6.container_memo...
2                 frontend-2                  60                  1647736260               inf  [{'kpi_name': 'container.node-5.container_memo...
3                frontend2-0                  57                  1647736200         78.125000  [{'kpi_name': 'mesh.destination.unknown.fronte...
4                  adservice                  42                  1647736320               inf  [{'kpi_name': 'runtime.java_lang_Memory_Object...
5    productcatalogservice-1                  36                  1647736560          1.869276  [{'kpi_name': 'container.node-6.container_netw...
6    productcatalogservice-2                  34                  1647736500               inf  [{'kpi_name': 'mesh.destination.recommendation...
7    productcatalogservice-0                  32                  1647736260               inf  [{'kpi_name': 'mesh.destination.recommendation...
8          checkoutservice-0                  32                  1647736260         11.619048  [{'kpi_name': 'mesh.source.checkoutservice.jae...
9                     node-5                  29                  1647736200          5.176471  [{'kpi_name': 'system.io.rkb_s', 'ratio': 5.17...
10             cartservice-0                  28                  1647736260               inf  [{'kpi_name': 'container.node-6.container_cpu_...
11               adservice-0                  26                  1647736560               inf  [{'kpi_name': 'container.node-6.container_fs_r...
12               adservice-2                  25                  1647736200               inf  [{'kpi_name': 'container.node-5.container_fs_r...
13                    node-4                  25                  1647736260               inf  [{'kpi_name': 'system.io.avg_q_sz', 'ratio': i...
14                    node-1                  25                  1647736200         33.611111  [{'kpi_name': 'system.net.tcp.retrans_segs', '...
15  productcatalogservice2-0                  25                  1647736440          3.539683  [{'kpi_name': 'mesh.source.productcatalogservi...
16              adservice2-0                  24                  1647736200               inf  [{'kpi_name': 'container.node-6.container_fs_r...
17   recommendationservice-1                  24                  1647736260          8.666667  [{'kpi_name': 'mesh.source.recommendationservi...
18         currencyservice-0                  24                  1647736260          1.266667  [{'kpi_name': 'container.node-6.container_memo...
19            cartservice2-0                  23                  1647736260               inf  [{'kpi_name': 'container.node-5.container_fs_r..., 54)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse existing df_metric and global_p95 if present, otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

try:
    global_p95
except NameError:
    global_p95 = (
        df_metric
        .dropna(subset=['value'])
        .groupby(['cmdb_id','kpi_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'global_P95'})
    )

# incident window in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2022,3,20,8,30,0))
end_dt   = tz.localize(datetime(2022,3,20,9,0,0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

# target components
components = ['frontend-0','frontend-1','frontend-2','node-5','node-6']

# filter metric data to window and components
df_win = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts) & (df_metric['cmdb_id'].isin(components))].copy()

# merge global P95 (computed globally) into the windowed data
df_win = df_win.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# drop rows missing value or global_P95
df_win = df_win.dropna(subset=['value','global_P95']).copy()

# compute ratio with safe handling for global_P95 == 0
df_win['global_P95_float'] = df_win['global_P95'].astype(float)
df_win['ratio'] = np.where(
    df_win['global_P95_float'] == 0,
    np.where(df_win['value'] > 0, np.inf, 0.0),
    df_win['value'] / df_win['global_P95_float']
)
df_win['is_anomaly'] = df_win['value'] > df_win['global_P95_float']

# keep only anomaly rows
df_anom = df_win[df_win['is_anomaly']].copy()

# prepare list to collect segment rows
segments_list = []

for comp in components:
    comp_anom = df_anom[df_anom['cmdb_id'] == comp].copy()
    if comp_anom.empty:
        continue
    # unique sorted timestamps (minute-granular)
    unique_ts = np.sort(comp_anom['timestamp'].unique())
    ts_df = pd.DataFrame({'timestamp': unique_ts})
    ts_df['diff'] = ts_df['timestamp'].diff().fillna(9999)
    # consecutive if diff == 60
    ts_df['new_seg'] = (ts_df['diff'] > 60).astype(int)
    ts_df['seg_id'] = ts_df['new_seg'].cumsum()
    # merge seg_id back to anomaly rows by timestamp
    comp_anom = comp_anom.merge(ts_df[['timestamp','seg_id']], on='timestamp', how='left')
    # aggregate per segment
    seg_agg = (
        comp_anom
        .groupby('seg_id')
        .agg(
            start_timestamp = ('timestamp','min'),
            end_timestamp = ('timestamp','max'),
            distinct_kpi_count_in_segment = ('kpi_name', 'nunique'),
            max_breach_ratio_in_segment = ('ratio', 'max')
        )
        .reset_index(drop=True)
    )
    # compute duration_minutes inclusive
    seg_agg['duration_minutes'] = ((seg_agg['end_timestamp'] - seg_agg['start_timestamp']) // 60) + 1
    # round ratios but keep inf
    def rformat(x):
        if pd.isna(x):
            return x
        if np.isinf(x):
            return np.inf
        return float(np.round(x,6))
    seg_agg['max_breach_ratio_in_segment'] = seg_agg['max_breach_ratio_in_segment'].map(rformat)
    # attach component column and sort by start timestamp
    seg_agg['component'] = comp
    seg_agg = seg_agg.sort_values('start_timestamp').reset_index(drop=True)
    # keep up to top 5 segments earliest first
    seg_agg = seg_agg.head(5)
    # reorder columns as requested
    seg_agg = seg_agg[['component','start_timestamp','end_timestamp','duration_minutes','distinct_kpi_count_in_segment','max_breach_ratio_in_segment']]
    segments_list.append(seg_agg)

# combine results, limit to max 25 rows
if segments_list:
    segments_df = pd.concat(segments_list, axis=0, ignore_index=True)
    segments_df = segments_df.head(25)
else:
    segments_df = pd.DataFrame(columns=['component','start_timestamp','end_timestamp','duration_minutes','distinct_kpi_count_in_segment','max_breach_ratio_in_segment'])

# display compact result
segments_df

```
Out[4]:
```
import pytz
from datetime import datetime

# build a plain-English summary from the previously computed segments_df
try:
    segments_df
except NameError:
    summary_text = "No segment data available."
else:
    tz = pytz.timezone('Asia/Shanghai')
    lines = []
    lines.append("Component-level fault segments within 2022-03-20 08:30:00–09:00:00 (UTC+8):\n")
    for comp in ['frontend-0','frontend-1','frontend-2','node-5','node-6']:
        comp_rows = segments_df[segments_df['component'] == comp]
        if comp_rows.empty:
            lines.append(f"- {comp}: no anomalous-minute segments in the window.")
            continue
        lines.append(f"- {comp}: {len(comp_rows)} segment(s).")
        for _, r in comp_rows.sort_values('start_timestamp').iterrows():
            start_ts = int(r['start_timestamp'])
            end_ts = int(r['end_timestamp'])
            start_human = datetime.fromtimestamp(start_ts, tz).strftime('%Y-03-20 %H:%M:%S')
            end_human = datetime.fromtimestamp(end_ts, tz).strftime('%Y-03-20 %H:%M:%S')
            dur = int(r['duration_minutes'])
            distinct_kpis = int(r['distinct_kpi_count_in_segment'])
            ratio = r['max_breach_ratio_in_segment']
            ratio_str = "inf" if (ratio == float('inf')) else f"{float(ratio):.6f}"
            lines.append(f"    • Segment: {start_ts} ({start_human}) → {end_ts} ({end_human}), duration {dur} min, {distinct_kpis} distinct KPIs breached, max breach ratio {ratio_str}.")
    summary_text = "\n".join(lines)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

component  start_timestamp  end_timestamp  duration_minutes  distinct_kpi_count_in_segment  max_breach_ratio_in_segment
0   frontend-0       1647736200     1647736260                 2                             20                     1.038462
1   frontend-0       1647736500     1647736620                 3                             29                          inf
2   frontend-0       1647736800     1647737040                 5                              7                     6.315146
3   frontend-0       1647737160     1647737160                 1                              2                     1.052632
4   frontend-0       1647737340     1647737700                 7                             10                     4.196309
..         ...              ...            ...               ...                            ...                          ...
19      node-5       1647737100     1647737280                 4                              8                     2.666667
20      node-6       1647736260     1647736320                 2                              4                     1.480392
21      node-6       1647736560     1647736560                 1                              4                          inf
22      node-6       1647737520     1647737520                 1                              3                     1.500000
23      node-6       1647738000     1647738000                 1                             13                     1.075630

[24 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# ensure types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) using full series (do NOT filter first)
trace_group_keys = ['cmdb_id', 'trace_name']
global_p95_trace = (
    df_trace
    .dropna(subset=['value'])
    .groupby(trace_group_keys)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)

# helper to analyze a given time window (inclusive)
def analyze_trace_window(df_trace, global_p95_trace, window_start_ts, window_end_ts, top_n_components=20):
    # filter window
    df_win = df_trace[(df_trace['timestamp'] >= window_start_ts) & (df_trace['timestamp'] <= window_end_ts)].copy()
    if df_win.shape[0] == 0:
        return pd.DataFrame(columns=['cmdb_id','distinct_trace_names_count','earliest_anomaly_timestamp_in_window','max_breach_ratio','top_trace_names'])
    # merge global P95 (computed globally)
    df_win = df_win.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')
    # drop missing values
    df_win = df_win.dropna(subset=['value','global_P95']).copy()
    if df_win.shape[0] == 0:
        return pd.DataFrame(columns=['cmdb_id','distinct_trace_names_count','earliest_anomaly_timestamp_in_window','max_breach_ratio','top_trace_names'])
    # compute ratio and anomaly flag (handle global_P95 == 0)
    df_win['global_P95_f'] = df_win['global_P95'].astype(float)
    df_win['ratio'] = np.where(
        df_win['global_P95_f'] == 0,
        np.where(df_win['value'] > 0, np.inf, 0.0),
        df_win['value'] / df_win['global_P95_f']
    )
    df_win['is_anomaly'] = df_win['value'] > df_win['global_P95_f']
    # keep only anomalous points
    df_anom = df_win[df_win['is_anomaly']].copy()
    if df_anom.shape[0] == 0:
        return pd.DataFrame(columns=['cmdb_id','distinct_trace_names_count','earliest_anomaly_timestamp_in_window','max_breach_ratio','top_trace_names'])
    # per (cmdb_id, trace_name) compute max ratio and earliest anomaly timestamp in window
    per_trace = (
        df_anom
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(max_ratio=('ratio','max'),
             earliest_trace_anom_ts=('timestamp','min'))
    )
    # per component aggregations
    distinct_count = per_trace.groupby('cmdb_id')['trace_name'].nunique().rename('distinct_trace_names_count')
    earliest_comp_ts = per_trace.groupby('cmdb_id')['earliest_trace_anom_ts'].min().rename('earliest_anomaly_timestamp_in_window')
    max_breach = per_trace.groupby('cmdb_id')['max_ratio'].max().rename('max_breach_ratio')
    comp_stats = pd.concat([distinct_count, earliest_comp_ts, max_breach], axis=1).reset_index()
    # compute top_trace_names per component (up to 3) by per-trace max_ratio
    per_trace_sorted = per_trace.sort_values(['cmdb_id','max_ratio'], ascending=[True, False]).copy()
    top_traces = (
        per_trace_sorted
        .groupby('cmdb_id')
        .apply(lambda d: d.head(3)[['trace_name','max_ratio']].to_dict(orient='records'))
        .rename('top_trace_names')
        .reset_index()
    )
    comp_stats = comp_stats.merge(top_traces, on='cmdb_id', how='left')
    # format ratios (round, keep inf)
    def format_ratio(x):
        if pd.isna(x):
            return x
        if np.isinf(x):
            return np.inf
        return float(np.round(x,6))
    comp_stats['max_breach_ratio'] = comp_stats['max_breach_ratio'].map(format_ratio)
    # format top trace ratios similarly
    def fmt_top(list_objs):
        if not isinstance(list_objs, list):
            return []
        out = []
        for it in list_objs:
            r = it.get('max_ratio', None)
            out.append({'trace_name': it.get('trace_name'), 'ratio': (np.inf if (r is not None and np.isinf(r)) else (None if r is None else float(np.round(r,6))))})
        return out
    comp_stats['top_trace_names'] = comp_stats['top_trace_names'].map(fmt_top)
    # sort by distinct_trace_names_count desc then max_breach_ratio desc and limit
    comp_stats = comp_stats.sort_values(by=['distinct_trace_names_count','max_breach_ratio'], ascending=[False, False]).head(top_n_components)
    # ensure earliest timestamp is Int64
    comp_stats['earliest_anomaly_timestamp_in_window'] = comp_stats['earliest_anomaly_timestamp_in_window'].astype('Int64')
    # select columns in requested order
    out = comp_stats[['cmdb_id','distinct_trace_names_count','earliest_anomaly_timestamp_in_window','max_breach_ratio','top_trace_names']].reset_index(drop=True)
    return out

# Define the two windows
A_start, A_end = 1647736200, 1647736320
B_start, B_end = 1647737332, 1647737513

result_A = analyze_trace_window(df_trace, global_p95_trace, A_start, A_end, top_n_components=20)
result_B = analyze_trace_window(df_trace, global_p95_trace, B_start, B_end, top_n_components=20)

# Compact display: two DataFrames (top components per window)
result_A, result_B

```
Out[5]:
```
Summary of trace-based anomaly analysis (global P95 per (cmdb_id, trace_name) used):

Window A — Early window (1647736200 → 1647736320 UTC+8; 2022-03-20 08:30:00–08:32:00)
- Components with trace metrics > global P95: several; top results (sorted by number of distinct breached trace_names):
  1. frontend-2 — 12 distinct trace_names breached; earliest breach 1647736200 (2022-03-20 08:30:00); max breach ratio 1.512605. Top trace examples: trace.to_adservice-2.duration... (highest ratio).
  2. currencyservice-1 — 6 distinct; earliest 1647736200; max ratio 1.081861. Top: trace.from_frontend-1.duration...
  3. checkoutservice-0 — 4 distinct; earliest 1647736260 (08:31:00); max ratio 1.097046. Top: trace.to_paymentservice-2.duration...
  4. checkoutservice-1 — 4 distinct; earliest 1647736200; max ratio 1.078806.
  5. productcatalogservice-0 — 4 distinct; earliest 1647736200; max ratio 1.006513.
  (Other components with smaller counts include frontend-1, checkoutservice-2, currencyservice2-0, frontend2-0, adservice-2, paymentservice-2, etc.)
- Highest per-component breach ratio in this window is ~2.419461 (currencyservice2-0 and frontend2-0), while many components show ratios near ~1.01–1.51.

Window B — Error-log window (1647737332 → 1647737513 UTC+8; ~2022-03-20 08:48:52–08:51:53)
- Components with trace metrics > global P95: several; top results:
  1. frontend-0 — 11 distinct trace_names breached; earliest breach 1647737340 (~08:49:00); max breach ratio 1.454545. Top trace examples: trace.to_adservice-2.duration...
  2. checkoutservice-2 — 10 distinct; earliest 1647737340; max breach ratio 5.615942 (largest ratio observed in this window).
  3. currencyservice-2 — 7 distinct; earliest 1647737340; max ratio 1.180168.
  4. productcatalogservice-2 — 5 distinct; earliest 1647737400 (~08:50:00); max ratio 1.125541.
  5. recommendationservice2-0 — 4 distinct; earliest 1647737340; max ratio 1.125541.
  (Other notable entries: checkoutservice-0, currencyservice-0 (ratio 5.615942), productcatalogservice-1, frontend-2, frontend2-0, etc.)
- The largest breach ratios in this error-log window reach ~5.62 (checkoutservice-2 and currencyservice-0), indicating stronger deviations from historical P95 during the error-log interval.

Overall interpretation
- Early window (08:30–08:32) anomalies concentrate on frontend-2 and several checkout/currency/productcatalog components with moderate breach ratios (mostly ~1.0–2.4).
- Error-log window (~08:48–08:51) shows a different/expanded set of components affected, with frontend-0 and checkoutservice-2 among the top impacted, and the largest breach ratios (up to ~5.6) occurring in this interval.
- Top_trace_names fields (in results) identify which specific trace metrics contributed the breaches for each component.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  distinct_trace_names_count  earliest_anomaly_timestamp_in_window  max_breach_ratio                                    top_trace_names
0                frontend-2                          12                            1647736200          1.512605  [{'trace_name': 'trace.to_adservice-2.duration...
1         currencyservice-1                           6                            1647736200          1.081861  [{'trace_name': 'trace.from_frontend-1.duratio...
2         checkoutservice-0                           4                            1647736260          1.097046  [{'trace_name': 'trace.to_paymentservice-2.dur...
3         checkoutservice-1                           4                            1647736200          1.078806  [{'trace_name': 'trace.to_productcatalogservic...
4   productcatalogservice-0                           4                            1647736200          1.006513  [{'trace_name': 'trace.from_frontend-1.duratio...
5                frontend-1                           3                            1647736200          1.081861  [{'trace_name': 'trace.to_currencyservice-1.du...
6         checkoutservice-2                           3                            1647736320          1.010138  [{'trace_name': 'trace.to_emailservice-0.durat...
7        currencyservice2-0                           2                            1647736200          2.419461  [{'trace_name': 'trace.from_frontend2-0.durati...
8               frontend2-0                           2                            1647736200          2.419461  [{'trace_name': 'trace.to_currencyservice2-0.d...
9               adservice-2                           2                            1647736200          1.512605  [{'trace_name': 'trace.from_frontend-2.duratio...
10         paymentservice-2                           2                            1647736260          1.097046  [{'trace_name': 'trace.from_checkoutservice-0....
11              adservice-1                           2                            1647736320          1.086262  [{'trace_name': 'trace.from_frontend-2.duratio...
12  productcatalogservice-1                           2                            1647736200          1.078806  [{'trace_name': 'trace.from_checkoutservice-1....
13        shippingservice-1                           2                            1647736200          1.040100  [{'trace_name': 'trace.from_checkoutservice-1....
14               frontend-0                           2                            1647736200          1.022167  [{'trace_name': 'trace.to_currencyservice-1.du...
15  recommendationservice-0                           2                            1647736200          1.012410  [{'trace_name': 'trace.from_frontend-2.duratio...
16  productcatalogservice-2                           2                            1647736320          1.010576  [{'trace_name': 'trace.from_recommendationserv...
17           emailservice-0                           2                            1647736320          1.010138  [{'trace_name': 'trace.from_checkoutservice-2....
18  recommendationservice-1                           2                            1647736320          1.006293  [{'trace_name': 'trace.from_frontend-2.duratio...
19        currencyservice-2                           1                            1647736200          1.027586  [{'trace_name': 'trace.from_frontend-2.duratio...,                      cmdb_id  distinct_trace_names_count  earliest_anomaly_timestamp_in_window  max_breach_ratio                                    top_trace_names
0                 frontend-0                          11                            1647737340          1.454545  [{'trace_name': 'trace.to_adservice-2.duration...
1          checkoutservice-2                          10                            1647737340          5.615942  [{'trace_name': 'trace.to_currencyservice-0.du...
2          currencyservice-2                           7                            1647737340          1.180168  [{'trace_name': 'trace.from_checkoutservice-1....
3    productcatalogservice-2                           5                            1647737400          1.125541  [{'trace_name': 'trace.from_recommendationserv...
4   recommendationservice2-0                           4                            1647737340          1.125541  [{'trace_name': 'trace.to_productcatalogservic...
5          checkoutservice-0                           4                            1647737400          1.102151  [{'trace_name': 'trace.to_currencyservice-1.du...
6          currencyservice-0                           3                            1647737340          5.615942  [{'trace_name': 'trace.from_checkoutservice-2....
7    productcatalogservice-1                           3                            1647737340          1.714697  [{'trace_name': 'trace.from_recommendationserv...
8                 frontend-2                           3                            1647737340          1.155250  [{'trace_name': 'trace.self.error_rate', 'rati...
9          currencyservice-1                           3                            1647737400          1.102151  [{'trace_name': 'trace.from_checkoutservice-0....
10   productcatalogservice-0                           3                            1647737340          1.064516  [{'trace_name': 'trace.from_recommendationserv...
11         shippingservice-0                           2                            1647737400          4.606599  [{'trace_name': 'trace.from_checkoutservice-2....
12               frontend2-0                           2                            1647737400          2.262409  [{'trace_name': 'trace.self.error_rate', 'rati...
13        checkoutservice2-0                           2                            1647737340          2.019765  [{'trace_name': 'trace.to_currencyservice2-0.d...
14        currencyservice2-0                           2                            1647737340          2.019765  [{'trace_name': 'trace.from_checkoutservice2-0...
15   recommendationservice-1                           2                            1647737400          1.714697  [{'trace_name': 'trace.to_productcatalogservic...
16               adservice-2                           2                            1647737400          1.454545  [{'trace_name': 'trace.from_frontend-0.duratio...
17                frontend-1                           2                            1647737340          1.384263  [{'trace_name': 'trace.self.error_rate', 'rati...
18         checkoutservice-1                           2                            1647737400          1.180168  [{'trace_name': 'trace.to_currencyservice-2.du...
19               adservice-0                           2                            1647737400          1.089109  [{'trace_name': 'trace.from_frontend-0.duratio...)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import re

# reuse existing df_trace and global_p95_trace if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

try:
    global_p95_trace
except NameError:
    global_p95_trace = (
        df_trace
        .dropna(subset=['value'])
        .groupby(['cmdb_id','trace_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'global_P95'})
    )

# Load error_logs.csv
df_errors = pd.read_csv('error_logs.csv')
df_errors['timestamp'] = df_errors['timestamp'].astype(int)

# Window boundaries (inclusive)
A_start, A_end = 1647736200, 1647736320
B_start, B_end = 1647737332, 1647737513

# 1) error_logs summary for frontend pods in window B
frontends = ['frontend-0','frontend-1','frontend-2']
err_win = df_errors[(df_errors['timestamp'] >= B_start) & (df_errors['timestamp'] <= B_end) & (df_errors['cmdb_id'].isin(frontends))].copy()

error_summary_list = []
for cm in frontends:
    sub = err_win[err_win['cmdb_id'] == cm]
    if sub.shape[0] == 0:
        # still include pod with zero counts as empty result per instruction to limit to three frontend pods only
        entry = {
            "cmdb_id": cm,
            "message_count": 0,
            "earliest_timestamp": None,
            "latest_timestamp": None,
            "top_messages": []
        }
    else:
        msg_counts = sub['message'].value_counts().head(3)
        top_msgs = [{"message": str(m), "count": int(c)} for m, c in msg_counts.items()]
        entry = {
            "cmdb_id": cm,
            "message_count": int(sub.shape[0]),
            "earliest_timestamp": int(sub['timestamp'].min()),
            "latest_timestamp": int(sub['timestamp'].max()),
            "top_messages": top_msgs
        }
    error_summary_list.append(entry)

# 2) trace-breach breakdown for both windows
# prepare merged trace with global P95
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_t = df_trace.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')
df_t = df_t.dropna(subset=['value','global_P95']).copy()
df_t['global_P95_f'] = df_t['global_P95'].astype(float)
# compute ratio and anomaly flag
df_t['ratio'] = np.where(
    df_t['global_P95_f'] == 0,
    np.where(df_t['value'] > 0, np.inf, 0.0),
    df_t['value'] / df_t['global_P95_f']
)
df_t['is_anomaly'] = df_t['value'] > df_t['global_P95_f']

def extract_target_service(trace_name):
    # match trace.to_<service> or trace.from_<service>
    m = re.search(r'trace\.(?:to|from)_([A-Za-z0-9\-\_]+)', trace_name)
    return m.group(1) if m else None

def summarize_window(win_start, win_end, top_n=10):
    win = df_t[(df_t['timestamp'] >= win_start) & (df_t['timestamp'] <= win_end) & (df_t['is_anomaly'])].copy()
    if win.shape[0] == 0:
        return []
    # group by trace_name
    grp = win.groupby('trace_name')
    summary = (
        grp.agg(
            breach_count = ('trace_name','count'),
            sample_cmdb_id = ('cmdb_id', lambda s: s.iloc[0]),
            earliest_anomaly_timestamp_in_window = ('timestamp','min'),
            max_breach_ratio = ('ratio','max')
        )
        .reset_index()
    )
    # extract target service
    summary['extracted_target_service'] = summary['trace_name'].map(extract_target_service)
    # format ratios and timestamps; convert inf to string "inf" for JSON-compatibility
    def fmt_ratio(x):
        if pd.isna(x):
            return None
        if np.isinf(x):
            return "inf"
        return float(np.round(x,6))
    summary['max_breach_ratio'] = summary['max_breach_ratio'].map(fmt_ratio)
    summary['earliest_anomaly_timestamp_in_window'] = summary['earliest_anomaly_timestamp_in_window'].astype(int)
    summary['breach_count'] = summary['breach_count'].astype(int)
    # sort by breach_count desc then max_breach_ratio desc (treat "inf" as highest)
    def sort_key(row):
        ratio = row['max_breach_ratio']
        # convert "inf" to very large number for sorting
        rsort = float('inf') if ratio == "inf" else (ratio if ratio is not None else -1.0)
        return (row['breach_count'], rsort)
    summary = summary.sort_values(by=['breach_count','max_breach_ratio'], ascending=[False, False])
    summary = summary.head(top_n)
    # build compact list of dicts
    out = []
    for _, r in summary.iterrows():
        out.append({
            "trace_name": r['trace_name'],
            "sample_cmdb_id": r['sample_cmdb_id'],
            "breach_count": int(r['breach_count']),
            "earliest_anomaly_timestamp_in_window": int(r['earliest_anomaly_timestamp_in_window']),
            "max_breach_ratio": r['max_breach_ratio'],
            "extracted_target_service": r['extracted_target_service'] if pd.notna(r['extracted_target_service']) else None
        })
    return out

window_A_list = summarize_window(A_start, A_end, top_n=10)
window_B_list = summarize_window(B_start, B_end, top_n=10)

final_result = {
    "error_logs_frontend_summary": error_summary_list,
    "trace_breach_breakdown": {
        "window_A": window_A_list,
        "window_B": window_B_list
    }
}

final_result

```
Out[6]:
```
summary = (
    "Results summary (compact):\n\n"
    "1) Error logs (window 1647737332–1647737513): Frontend pods\n"
    "- frontend-0: message_count=65, earliest=1647737334, latest=1647737512. Top message: "
    "'severity: warning, message: failed to retrieve ads' (65 occurrences).\n"
    "- frontend-1: message_count=66, earliest=1647737332, latest=1647737506. Top message: "
    "'severity: warning, message: failed to retrieve ads' (66 occurrences).\n"
    "- frontend-2: message_count=68, earliest=1647737333, latest=1647737513. Top message: "
    "'severity: warning, message: failed to retrieve ads' (68 occurrences).\n\n"
    "2) Trace-breach breakdown — top breached trace_names per window (compact):\n\n"
    "Window A — Early (1647736200–1647736320): top items\n"
    "- trace.from_frontend-2.duration_p95: sample_cmdb_id=adservice-2, breach_count=6, "
    "earliest=1647736200, max_breach_ratio=1.512605 (target service: frontend-2).\n"
    "- trace.from_frontend-2.duration_mean: sample_cmdb_id=adservice-2, breach_count=6, "
    "earliest=1647736200, max_breach_ratio=1.196581 (target: frontend-2).\n"
    "- trace.to_currencyservice-1.duration_p95: sample_cmdb_id=frontend-1, breach_count=4, "
    "earliest=1647736200, max_breach_ratio=1.063554 (target: currencyservice-1).\n"
    "- trace.from_frontend2-0.duration_mean / trace.to_currencyservice2-0.duration_mean: "
    "each breach_count=2, earliest=1647736200, max_breach_ratio≈1.215511 (targets: frontend2-0 / currencyservice2-0).\n"
    "- Additional top entries include traces from checkoutservice and productcatalogservice with smaller counts/ratios.\n\n"
    "Window B — Error-log (1647737332–1647737513): top items\n"
    "- trace.self.error_rate: sample_cmdb_id=frontend-0, breach_count=8, earliest=1647737340, "
    "max_breach_ratio=2.262409 (no target service extracted).\n"
    "- trace.from_frontend-0.duration_p95: sample_cmdb_id=currencyservice-2, breach_count=6, "
    "earliest=1647737340, max_breach_ratio=1.454545 (target: frontend-0).\n"
    "- trace.from_checkoutservice-2.duration_mean: sample_cmdb_id=currencyservice-0, breach_count=4, "
    "earliest=1647737340, max_breach_ratio=5.615942 (target: checkoutservice-2) — notable large deviation.\n"
    "- trace.from_frontend-0.duration_mean and trace.to_currencyservice-2.duration_p95 also appear among top breaches (breach_count=4, max ratios ~1.35 and ~1.17 respectively).\n"
    "- Other top entries include traces to productcatalogservice and currencyservice with smaller counts/ratios.\n\n"
    "Overall: Error logs show consistent 'failed to retrieve ads' warnings across all three frontends during the error-log window. "
    "Trace breaches shift between the early window (frontend-2 related traces prominent) and the error-log window (frontend-0 and checkoutservice-2 traces prominent), "
    "with the largest trace deviation observed in the error-log window (ratio ~5.62 for checkoutservice-2 -> currencyservice trace)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'error_logs_frontend_summary': [{'cmdb_id': 'frontend-0', 'message_count': 65, 'earliest_timestamp': 1647737334, 'latest_timestamp': 1647737512, 'top_messages': [{'message': 'severity: warning, message: failed to retrieve ads', 'count': 65}]}, {'cmdb_id': 'frontend-1', 'message_count': 66, 'earliest_timestamp': 1647737332, 'latest_timestamp': 1647737506, 'top_messages': [{'message': 'severity: warning, message: failed to retrieve ads', 'count': 66}]}, {'cmdb_id': 'frontend-2', 'message_count': 68, 'earliest_timestamp': 1647737333, 'latest_timestamp': 1647737513, 'top_messages': [{'message': 'severity: warning, message: failed to retrieve ads', 'count': 68}]}], 'trace_breach_breakdown': {'window_A': [{'trace_name': 'trace.from_frontend-2.duration_p95', 'sample_cmdb_id': 'adservice-2', 'breach_count': 6, 'earliest_anomaly_timestamp_in_window': 1647736200, 'max_breach_ratio': 1.512605, 'extracted_target_service': 'frontend-2'}, {'trace_name': 'trace.from_frontend-2.duration_mean', 'sample_cmdb_id': 'adservice-2', 'breach_count': 6, 'earliest_anomaly_timestamp_in_window': 1647736200, 'max_breach_ratio': 1.196581, 'extracted_target_service': 'frontend-2'}, {'trace_name': 'trace.to_currencyservice-1.duration_p95', 'sample_cmdb_id': 'frontend-1', 'breach_count': 4, 'earliest_anomaly_timestamp_in_window': 1647736200, 'max_breach_ratio': 1.063554, 'extracted_target_service': 'currencyservice-1'}, {'trace_name': 'trace.from_frontend2-0.duration_mean', 'sample_cmdb_id': 'currencyservice2-0', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647736200, 'max_breach_ratio': 1.215511, 'extracted_target_service': 'frontend2-0'}, {'trace_name': 'trace.to_currencyservice2-0.duration_mean', 'sample_cmdb_id': 'frontend2-0', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647736200, 'max_breach_ratio': 1.215511, 'extracted_target_service': 'currencyservice2-0'}, {'trace_name': 'trace.from_checkoutservice-0.duration_mean', 'sample_cmdb_id': 'paymentservice-2', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647736260, 'max_breach_ratio': 1.097046, 'extracted_target_service': 'checkoutservice-0'}, {'trace_name': 'trace.from_checkoutservice-0.duration_p95', 'sample_cmdb_id': 'paymentservice-2', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647736260, 'max_breach_ratio': 1.097046, 'extracted_target_service': 'checkoutservice-0'}, {'trace_name': 'trace.to_currencyservice-1.duration_mean', 'sample_cmdb_id': 'frontend-1', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647736200, 'max_breach_ratio': 1.081861, 'extracted_target_service': 'currencyservice-1'}, {'trace_name': 'trace.from_checkoutservice-1.duration_mean', 'sample_cmdb_id': 'productcatalogservice-1', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647736200, 'max_breach_ratio': 1.078806, 'extracted_target_service': 'checkoutservice-1'}, {'trace_name': 'trace.from_frontend-1.duration_p95', 'sample_cmdb_id': 'currencyservice-1', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647736200, 'max_breach_ratio': 1.063554, 'extracted_target_service': 'frontend-1'}], 'window_B': [{'trace_name': 'trace.self.error_rate', 'sample_cmdb_id': 'frontend-0', 'breach_count': 8, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 2.262409, 'extracted_target_service': None}, {'trace_name': 'trace.from_frontend-0.duration_p95', 'sample_cmdb_id': 'currencyservice-2', 'breach_count': 6, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 1.454545, 'extracted_target_service': 'frontend-0'}, {'trace_name': 'trace.from_checkoutservice-2.duration_mean', 'sample_cmdb_id': 'currencyservice-0', 'breach_count': 4, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 5.615942, 'extracted_target_service': 'checkoutservice-2'}, {'trace_name': 'trace.from_frontend-0.duration_mean', 'sample_cmdb_id': 'currencyservice-0', 'breach_count': 4, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 1.350482, 'extracted_target_service': 'frontend-0'}, {'trace_name': 'trace.to_currencyservice-2.duration_p95', 'sample_cmdb_id': 'frontend-0', 'breach_count': 4, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 1.166322, 'extracted_target_service': 'currencyservice-2'}, {'trace_name': 'trace.from_checkoutservice-2.duration_p95', 'sample_cmdb_id': 'currencyservice-0', 'breach_count': 3, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 5.085302, 'extracted_target_service': 'checkoutservice-2'}, {'trace_name': 'trace.to_currencyservice-2.duration_mean', 'sample_cmdb_id': 'frontend-0', 'breach_count': 3, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 1.180168, 'extracted_target_service': 'currencyservice-2'}, {'trace_name': 'trace.to_productcatalogservice-2.duration_p95', 'sample_cmdb_id': 'checkoutservice-0', 'breach_count': 3, 'earliest_anomaly_timestamp_in_window': 1647737400, 'max_breach_ratio': 1.125541, 'extracted_target_service': 'productcatalogservice-2'}, {'trace_name': 'trace.to_currencyservice-0.duration_mean', 'sample_cmdb_id': 'checkoutservice-2', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 5.615942, 'extracted_target_service': 'currencyservice-0'}, {'trace_name': 'trace.to_productcatalogservice-1.duration_p95', 'sample_cmdb_id': 'frontend-0', 'breach_count': 2, 'earliest_anomaly_timestamp_in_window': 1647737340, 'max_breach_ratio': 1.714697, 'extracted_target_service': 'productcatalogservice-1'}]}}```
```

In [ ]:
# In[7]:

import re
import pandas as pd
import numpy as np
import pytz

# reuse existing df_metric and global_p95 if available, otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

try:
    global_p95
except NameError:
    global_p95 = (
        df_metric
        .dropna(subset=['value'])
        .groupby(['cmdb_id','kpi_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'global_P95'})
    )

# pods of interest
pods = [
 'frontend-0','frontend-1','frontend-2',
 'checkoutservice-2',
 'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2',
 'currencyservice-0','currencyservice-1','currencyservice-2',
 'cartservice-0','cartservice-1','cartservice-2',
 'recommendationservice-0','recommendationservice-1','recommendationservice-2',
 'adservice-0','adservice-1','adservice-2',
 'paymentservice-0','paymentservice-1','paymentservice-2'
]

# regex to extract node label like 'node-5' from kpi_name containing 'container.node-<N>'
node_pattern = re.compile(r'container\.node-(node-\d+)|container\.node-(\d+)')  # fallback
# simpler extractor: search 'container.node-<num>' and build 'node-<num>'
def extract_node(k):
    if not isinstance(k, str):
        return None
    m = re.search(r'container\.node-(\d+)', k)
    if m:
        return f'node-{m.group(1)}'
    return None

# Filter full series for rows for these pods with node-specific kpi_name
mask_full = df_metric['cmdb_id'].isin(pods) & df_metric['kpi_name'].str.contains(r'container\.node-\d+', na=False)
df_nodes_full = df_metric[mask_full].copy()
df_nodes_full['node'] = df_nodes_full['kpi_name'].map(extract_node)

# compute node_list and node_kpi_counts per pod across full series
node_kpi_counts_df = (
    df_nodes_full
    .dropna(subset=['node'])
    .groupby(['cmdb_id','node'], as_index=False)
    .size()
    .rename(columns={'size':'node_kpi_count'})
)
# pivot/aggregate into mappings per pod
node_list_series = node_kpi_counts_df.groupby('cmdb_id')['node'].apply(lambda s: sorted(s.unique())).to_dict()
node_kpi_counts_map = node_kpi_counts_df.groupby('cmdb_id').apply(lambda g: dict(zip(g['node'], g['node_kpi_count']))).to_dict()

# Early window boundaries
early_start, early_end = 1647736200, 1647736320

# Prepare windowed df for anomaly checking: keep only rows in early window for pods and node-specific KPIs
mask_win = df_metric['cmdb_id'].isin(pods) & df_metric['timestamp'].between(early_start, early_end) & df_metric['kpi_name'].str.contains(r'container\.node-\d+', na=False)
df_win_nodes = df_metric[mask_win].copy()
df_win_nodes['node'] = df_win_nodes['kpi_name'].map(extract_node)

# Merge global P95 thresholds (precomputed) for anomaly evaluation (do NOT recompute thresholds after filtering)
df_win_nodes = df_win_nodes.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')
# drop rows without threshold or value
df_win_nodes = df_win_nodes.dropna(subset=['value','global_P95']).copy()
# compute anomaly flag: value > global_P95 (use float)
df_win_nodes['global_P95_f'] = df_win_nodes['global_P95'].astype(float)
df_win_nodes['is_anomaly'] = df_win_nodes['value'] > df_win_nodes['global_P95_f']

# compute node_anomaly_count_in_early_window per pod and node
anom_counts_df = (
    df_win_nodes[df_win_nodes['is_anomaly']]
    .dropna(subset=['node'])
    .groupby(['cmdb_id','node'], as_index=False)
    .agg(node_anomaly_count_in_early_window=('is_anomaly','sum'),
         earliest_node_anomaly_timestamp_in_window=('timestamp','min'))
)

# build per-pod mappings for anomaly counts and earliest timestamp
anom_counts_map = {}
earliest_node_anom_ts_map = {}
for _, row in anom_counts_df.iterrows():
    pod = row['cmdb_id']
    node = row['node']
    cnt = int(row['node_anomaly_count_in_early_window'])
    ts = int(row['earliest_node_anomaly_timestamp_in_window'])
    anom_counts_map.setdefault(pod, {})[node] = cnt
    # track earliest per pod across nodes
    prev = earliest_node_anom_ts_map.get(pod)
    if prev is None or ts < prev:
        earliest_node_anom_ts_map[pod] = ts

# assemble final list of per-pod JSON-serializable objects
result_list = []
for pod in pods:
    node_list = node_list_series.get(pod, [])
    node_kpi_counts = node_kpi_counts_map.get(pod, {})
    node_anomaly_counts = anom_counts_map.get(pod, {})
    earliest_ts = earliest_node_anom_ts_map.get(pod, None)
    entry = {
        "cmdb_id": pod,
        "node_list": node_list,
        "node_kpi_counts": {k:int(v) for k,v in node_kpi_counts.items()},
        "node_anomaly_count_in_early_window": {k:int(v) for k,v in node_anomaly_counts.items()},
        "earliest_node_anomaly_timestamp_in_early_window": int(earliest_ts) if earliest_ts is not None else None
    }
    result_list.append(entry)

# limit to top 50 (pods list is smaller)
result_list = result_list[:50]

# display compact result variable
result_list

```
Out[7]:
```
Summary of pod → node mapping and node-specific KPI anomalies in the early window (1647736200 → 1647736320, i.e., 2022-03-20 08:30–08:32 UTC+8):

General notes
- For the examined pods most node-specific KPIs reference a single node per pod (either node-6 or node-5).
- The reported node_kpi_counts show 3904 node-specific KPI rows for each (pod, node) across the full metric series (same count for node-5 and node-6 entries listed).

Per-pod results (only listed pods; times shown as integer timestamps; human times in UTC+8 in parentheses)
- frontend-0
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (no node-specific anomalies)
  - earliest_node_anomaly_timestamp_in_early_window: null

- frontend-1
  - node_list: [node-5]
  - node_kpi_counts: {node-5: 3904}
  - node_anomaly_count_in_early_window: {node-5: 1}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736200 (2022-03-20 08:30:00)

- frontend-2
  - node_list: [node-5]
  - node_kpi_counts: {node-5: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- checkoutservice-2
  - node_list: [node-5]
  - node_kpi_counts: {node-5: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- productcatalogservice-0
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {node-6: 1}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736260 (2022-03-20 08:31:00)

- productcatalogservice-1
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- productcatalogservice-2
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- currencyservice-0
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {node-6: 1}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736260 (2022-03-20 08:31:00)

- currencyservice-1
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {node-6: 2}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736260 (2022-03-20 08:31:00)

- currencyservice-2
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {node-6: 2}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736200 (2022-03-20 08:30:00)

- cartservice-0
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- cartservice-1
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {node-6: 4}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736200 (2022-03-20 08:30:00)

- cartservice-2
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {node-6: 3}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736200 (2022-03-20 08:30:00)

- recommendationservice-0
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {node-6: 1}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736260 (2022-03-20 08:31:00)

- recommendationservice-1
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- recommendationservice-2
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- adservice-0
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- adservice-1
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- adservice-2
  - node_list: [node-5]
  - node_kpi_counts: {node-5: 3904}
  - node_anomaly_count_in_early_window: {node-5: 1}
  - earliest_node_anomaly_timestamp_in_early_window: 1647736200 (2022-03-20 08:30:00)

- paymentservice-0
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- paymentservice-1
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

- paymentservice-2
  - node_list: [node-6]
  - node_kpi_counts: {node-6: 3904}
  - node_anomaly_count_in_early_window: {} (none)
  - earliest_node_anomaly_timestamp_in_early_window: null

Interpretation / takeaway
- Many pods’ node-specific KPIs reference node-6; a smaller set reference node-5 (notably frontend-1, frontend-2, adservice-2, checkoutservice-2).
- Node-specific anomalies in the early window are concentrated on a subset of pods:
  - cartservice-1 (4 node-KPI anomalies), cartservice-2 (3), several currencyservice pods (currencyservice-1 and -2 with 2 each, currencyservice-0 with 1), plus single anomalies for frontend-1, productcatalogservice-0, recommendationservice-0, adservice-2.
- Earliest node-related anomalies mostly occurred at 1647736200 (08:30) or 1647736260 (08:31).
- Several pods mapped to a node show zero node-specific anomalies in the early window despite having many node-specific KPI rows overall.

If you want this output as a JSON file or need human-readable timestamps for every entry, I can provide that.

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'frontend-0', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'frontend-1', 'node_list': ['node-5'], 'node_kpi_counts': {'node-5': 3904}, 'node_anomaly_count_in_early_window': {'node-5': 1}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736200}, {'cmdb_id': 'frontend-2', 'node_list': ['node-5'], 'node_kpi_counts': {'node-5': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'checkoutservice-2', 'node_list': ['node-5'], 'node_kpi_counts': {'node-5': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'productcatalogservice-0', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {'node-6': 1}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736260}, {'cmdb_id': 'productcatalogservice-1', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'productcatalogservice-2', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'currencyservice-0', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {'node-6': 1}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736260}, {'cmdb_id': 'currencyservice-1', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {'node-6': 2}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736260}, {'cmdb_id': 'currencyservice-2', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {'node-6': 2}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736200}, {'cmdb_id': 'cartservice-0', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'cartservice-1', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {'node-6': 4}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736200}, {'cmdb_id': 'cartservice-2', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {'node-6': 3}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736200}, {'cmdb_id': 'recommendationservice-0', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {'node-6': 1}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736260}, {'cmdb_id': 'recommendationservice-1', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'recommendationservice-2', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'adservice-0', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'adservice-1', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'adservice-2', 'node_list': ['node-5'], 'node_kpi_counts': {'node-5': 3904}, 'node_anomaly_count_in_early_window': {'node-5': 1}, 'earliest_node_anomaly_timestamp_in_early_window': 1647736200}, {'cmdb_id': 'paymentservice-0', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'paymentservice-1', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}, {'cmdb_id': 'paymentservice-2', 'node_list': ['node-6'], 'node_kpi_counts': {'node-6': 3904}, 'node_anomaly_count_in_early_window': {}, 'earliest_node_anomaly_timestamp_in_early_window': None}]```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import re
import pytz

# reuse existing df_metric and global_p95 if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

try:
    global_p95
except NameError:
    global_p95 = (
        df_metric
        .dropna(subset=['value'])
        .groupby(['cmdb_id','kpi_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'global_P95'})
    )

# Early window
early_start, early_end = 1647736200, 1647736320

# Nodes list
nodes = [f'node-{i}' for i in range(1,7)]

# Pre-filter metric rows in the early window (to reduce work)
df_win = df_metric[(df_metric['timestamp'] >= early_start) & (df_metric['timestamp'] <= early_end)].copy()

# Helper: extract node number from kpi_name if present (e.g., 'container.node-5' -> 'node-5')
def extract_node_from_kpi(k):
    if not isinstance(k, str):
        return None
    m = re.search(r'container\.node-(\d+)', k)
    return f'node-{m.group(1)}' if m else None

# Helper: extract resource label from kpi_name
def extract_resource_label(k):
    if not isinstance(k, str):
        return None
    k_low = k.lower()
    if 'cpu' in k_low or 'cfs' in k_low or 'pct_usage' in k_low:
        return 'cpu'
    if 'memory' in k_low or 'mem' in k_low or 'rss' in k_low:
        return 'memory'
    if 'network' in k_low or '.net.' in k_low or 'transmit' in k_low or 'receive' in k_low or 'packets' in k_low or 'mb' in k_low:
        return 'network'
    if 'fs' in k_low or '/dev' in k_low or 'reads' in k_low or 'writes' in k_low or 'rkb' in k_low or 'wkb' in k_low:
        return 'fs'
    if 'io' in k_low or 'iops' in k_low or 'avg_q' in k_low:
        return 'io'
    # fallback: try to pick a token
    parts = re.split(r'[.\-_/]', k_low)
    for tok in ['cpu','memory','mem','net','network','fs','io','disk']:
        if tok in parts:
            return tok
    return 'other'

# Merge global P95 into window rows for anomaly checks later (do not recompute thresholds)
df_win = df_win.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Keep only candidate node-specific KPI rows: either contain 'container.node-<n>' OR contain 'system.'
mask_node_kpi = df_win['kpi_name'].str.contains(r'container\.node-\d+', na=False) | df_win['kpi_name'].str.contains(r'\bsystem\.', na=False)
df_win_node_kpis = df_win[mask_node_kpi].copy()

# Extract node and resource label
df_win_node_kpis['node_extracted'] = df_win_node_kpis['kpi_name'].map(extract_node_from_kpi)
df_win_node_kpis['resource'] = df_win_node_kpis['kpi_name'].map(extract_resource_label)

# Determine anomaly: value > global_P95 (use global_p95 merged)
df_win_node_kpis = df_win_node_kpis.dropna(subset=['value', 'global_P95']).copy()
df_win_node_kpis['global_P95_f'] = df_win_node_kpis['global_P95'].astype(float)
df_win_node_kpis['ratio'] = np.where(
    df_win_node_kpis['global_P95_f'] == 0,
    np.where(df_win_node_kpis['value'] > 0, np.inf, 0.0),
    df_win_node_kpis['value'] / df_win_node_kpis['global_P95_f']
)
df_win_node_kpis['is_anomaly'] = df_win_node_kpis['value'] > df_win_node_kpis['global_P95_f']

# We'll consider node-specific breaches for a node if:
# - kpi_name contains container.node-<N> (node_extracted == node)
# OR
# - kpi_name contains 'system.' and cmdb_id equals the node (system metrics for node itself)
# Build a DataFrame of candidate breached rows per node
rows = []
for node in nodes:
    # container.node-* rows referencing this node
    mask1 = df_win_node_kpis['node_extracted'] == node
    # system.* rows where cmdb_id == node
    mask2 = (df_win_node_kpis['kpi_name'].str.contains(r'\bsystem\.', na=False)) & (df_win_node_kpis['cmdb_id'] == node)
    df_node_rows = df_win_node_kpis[mask1 | mask2].copy()
    if df_node_rows.shape[0] == 0:
        rows.append((node, df_node_rows))
    else:
        rows.append((node, df_node_rows))

# Prepare results per node
result_nodes = []
for node, df_nr in rows:
    # filter to breached points
    df_breach = df_nr[df_nr['is_anomaly']].copy()
    if df_breach.shape[0] == 0:
        result_nodes.append({
            "node": node,
            "distinct_kpi_count_breached": 0,
            "earliest_breach_timestamp": None,
            "max_breach_ratio": None,
            "top_resources": [],
            "affected_pod_count": 0,
            "sample_affected_pods": []
        })
        continue

    # distinct kpi_name count
    distinct_kpi_count = int(df_breach['kpi_name'].nunique())

    # earliest breach timestamp
    earliest_ts = int(df_breach['timestamp'].min())

    # max breach ratio (keep inf as float('inf'))
    max_ratio_val = df_breach['ratio'].replace([np.inf], [np.inf]).max()
    max_ratio_out = (float(max_ratio_val) if not np.isinf(max_ratio_val) else float('inf')) if pd.notna(max_ratio_val) else None

    # resource-level max ratios
    res_grp = df_breach.groupby('resource')['ratio'].max().reset_index()
    # map ratios formatting: keep inf as float('inf'), else round
    def fmt_rat(x):
        if pd.isna(x):
            return None
        if np.isinf(x):
            return float('inf')
        return float(np.round(x,6))
    res_grp['ratio_fmt'] = res_grp['ratio'].map(fmt_rat)
    # pick top 3 resources by ratio desc
    res_top = res_grp.sort_values('ratio', ascending=False).head(3)
    top_resources = [{"resource": r, "ratio": (float(v) if v!=float('inf') else float('inf'))} for r,v in zip(res_top['resource'], res_top['ratio_fmt'])]

    # affected_pod_count: distinct cmdb_id that are pods (exclude node cmdb_ids)
    pod_rows = df_breach[~df_breach['cmdb_id'].isin(nodes)].copy()
    affected_pods = pod_rows['cmdb_id'].unique().tolist()
    affected_pod_count = int(len(affected_pods))

    # sample_affected_pods: up to 5 distinct pod cmdb_ids earliest by their earliest breach timestamp
    if affected_pod_count > 0:
        pod_first_ts = pod_rows.groupby('cmdb_id')['timestamp'].min().reset_index().sort_values('timestamp').head(5)
        sample_pods = pod_first_ts['cmdb_id'].tolist()
    else:
        sample_pods = []

    result_nodes.append({
        "node": node,
        "distinct_kpi_count_breached": distinct_kpi_count,
        "earliest_breach_timestamp": int(earliest_ts),
        "max_breach_ratio": (float(max_ratio_out) if max_ratio_out is not None and not np.isinf(max_ratio_out) else (float('inf') if np.isinf(max_ratio_out) else None)),
        "top_resources": top_resources,
        "affected_pod_count": affected_pod_count,
        "sample_affected_pods": sample_pods
    })

# Ensure output is exactly 6 rows (node-1..node-6), in order
node_ordered = []
node_map = {r['node']: r for r in result_nodes}
for n in nodes:
    node_ordered.append(node_map.get(n, {
        "node": n,
        "distinct_kpi_count_breached": 0,
        "earliest_breach_timestamp": None,
        "max_breach_ratio": None,
        "top_resources": [],
        "affected_pod_count": 0,
        "sample_affected_pods": []
    }))

# final result variable
node_level_summary = node_ordered

node_level_summary

```
Out[8]:
```
Compact summary of node-level anomalies in the early window (1647736200–1647736320, i.e., 2022-03-20 08:30–08:32 UTC+8):

- node-1
  - distinct_kpi_count_breached: 1
  - earliest_breach_timestamp: 1647736200
  - max_breach_ratio: 7.365217
  - top_resources: cpu (ratio ~7.365)
  - affected_pod_count: 0 (no pods observed with node-specific KPI breaches)
  - sample_affected_pods: []

- node-2
  - distinct_kpi_count_breached: 6
  - earliest_breach_timestamp: 1647736200
  - max_breach_ratio: 1.25
  - top_resources: io (1.25), cpu (≈1.210345), other (≈1.19863)
  - affected_pod_count: 0
  - sample_affected_pods: []

- node-3
  - distinct_kpi_count_breached: 2
  - earliest_breach_timestamp: 1647736260
  - max_breach_ratio: ≈1.158537
  - top_resources: cpu (≈1.158537)
  - affected_pod_count: 0
  - sample_affected_pods: []

- node-4
  - distinct_kpi_count_breached: 3
  - earliest_breach_timestamp: 1647736260
  - max_breach_ratio: 2.0
  - top_resources: io (2.0), other (≈1.05)
  - affected_pod_count: 0
  - sample_affected_pods: []

- node-5
  - distinct_kpi_count_breached: 15
  - earliest_breach_timestamp: 1647736200
  - max_breach_ratio: ≈16.91028
  - top_resources: network (~16.91), io (~2.439), fs (~1.294)
  - affected_pod_count: 4
  - sample_affected_pods (earliest 5): ['adservice-2', 'frontend-1', 'cartservice2-0', 'shippingservice-2']

- node-6
  - distinct_kpi_count_breached: 13
  - earliest_breach_timestamp: 1647736200
  - max_breach_ratio: ≈17.19201
  - top_resources: network (~17.19), cpu (~5.553), io (~1.480)
  - affected_pod_count: 14
  - sample_affected_pods (earliest 5): ['cartservice-1', 'cartservice-2', 'currencyservice-2', 'emailservice-1', 'frontend2-0']

Key takeaways:
- node-5 and node-6 are the hotspots: they show the largest number of distinct breached node-specific KPIs and the highest max breach ratios (≈16.9 and ≈17.2), primarily driven by network-related KPIs.
- Nodes 1–4 have far fewer breaches; node-1 shows a high CPU ratio for a single KPI but no affected pods in the early window.
- node-6 affects many pods (14 distinct pods breached node-specific KPIs), while node-5 affects fewer pods (4) in this early window.

The original code execution output of IPython Kernel is also provided below for reference:

[{'node': 'node-1', 'distinct_kpi_count_breached': 1, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 7.365217391304349, 'top_resources': [{'resource': 'cpu', 'ratio': 7.365217}], 'affected_pod_count': 0, 'sample_affected_pods': []}, {'node': 'node-2', 'distinct_kpi_count_breached': 6, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.25, 'top_resources': [{'resource': 'io', 'ratio': 1.25}, {'resource': 'cpu', 'ratio': 1.210345}, {'resource': 'other', 'ratio': 1.19863}], 'affected_pod_count': 0, 'sample_affected_pods': []}, {'node': 'node-3', 'distinct_kpi_count_breached': 2, 'earliest_breach_timestamp': 1647736260, 'max_breach_ratio': 1.1585365853658538, 'top_resources': [{'resource': 'cpu', 'ratio': 1.158537}], 'affected_pod_count': 0, 'sample_affected_pods': []}, {'node': 'node-4', 'distinct_kpi_count_breached': 3, 'earliest_breach_timestamp': 1647736260, 'max_breach_ratio': 2.0, 'top_resources': [{'resource': 'io', 'ratio': 2.0}, {'resource': 'other', 'ratio': 1.05}], 'affected_pod_count': 0, 'sample_affected_pods': []}, {'node': 'node-5', 'distinct_kpi_count_breached': 15, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 16.91028044787159, 'top_resources': [{'resource': 'network', 'ratio': 16.91028}, {'resource': 'io', 'ratio': 2.439024}, {'resource': 'fs', 'ratio': 1.294118}], 'affected_pod_count': 4, 'sample_affected_pods': ['adservice-2', 'frontend-1', 'cartservice2-0', 'shippingservice-2']}, {'node': 'node-6', 'distinct_kpi_count_breached': 13, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 17.192012309519644, 'top_resources': [{'resource': 'network', 'ratio': 17.192012}, {'resource': 'cpu', 'ratio': 5.552953}, {'resource': 'io', 'ratio': 1.480392}], 'affected_pod_count': 14, 'sample_affected_pods': ['cartservice-1', 'cartservice-2', 'currencyservice-2', 'emailservice-1', 'frontend2-0']}]```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import re

# reuse existing loaded dataframes if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    global_p95
except NameError:
    global_p95 = (
        df_metric
        .dropna(subset=['value'])
        .groupby(['cmdb_id','kpi_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'global_P95'})
    )
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    global_p95_trace
except NameError:
    global_p95_trace = (
        df_trace
        .dropna(subset=['value'])
        .groupby(['cmdb_id','trace_name'])['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'global_P95'})
    )

# ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# helper: extract node from kpi_name like 'container.node-5'
def extract_node_from_kpi(k):
    if not isinstance(k, str):
        return None
    m = re.search(r'container\.node-(\d+)', k)
    return f'node-{m.group(1)}' if m else None

# helper: format ratio for JSON-friendly output (use "inf" string for infinite)
def fmt_ratio_val(x):
    if pd.isna(x):
        return None
    if np.isinf(x):
        return "inf"
    return float(np.round(float(x), 6))

# Early window and error-log window
A_start, A_end = 1647736200, 1647736320
B_start, B_end = 1647737332, 1647737513

# -----------------------------------------------------------------------------
# 1) Early window top node-specific KPI breaches for node-6 and node-5
# -----------------------------------------------------------------------------
nodes_of_interest = ['node-6','node-5']

# merge global p95 into full metric to reuse thresholds
df_metric_with_p95 = df_metric.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# filter to early window and candidate node-specific KPIs: container.node-<N> OR system.* (we will include system.* when cmdb_id==node)
mask_time = df_metric_with_p95['timestamp'].between(A_start, A_end)
mask_node_kpi = df_metric_with_p95['kpi_name'].str.contains(r'container\.node-\d+', na=False) | df_metric_with_p95['kpi_name'].str.contains(r'\bsystem\.', na=False)
df_early_candidates = df_metric_with_p95[mask_time & mask_node_kpi].copy()

# extract node if present
df_early_candidates['node_extracted'] = df_early_candidates['kpi_name'].map(extract_node_from_kpi)

# determine anomaly using precomputed p95 (do NOT recompute)
df_early_candidates = df_early_candidates.dropna(subset=['value','global_P95']).copy()
df_early_candidates['global_P95_f'] = df_early_candidates['global_P95'].astype(float)
df_early_candidates['ratio'] = np.where(
    df_early_candidates['global_P95_f'] == 0,
    np.where(df_early_candidates['value'] > 0, np.inf, 0.0),
    df_early_candidates['value'] / df_early_candidates['global_P95_f']
)
df_early_candidates['is_anomaly'] = df_early_candidates['value'] > df_early_candidates['global_P95_f']

# For each node of interest, select relevant rows:
# - rows where node_extracted == node (container.node-N referencing this node)
# - OR rows where kpi_name contains 'system.' AND cmdb_id == node (system metrics for node itself)
node_kpi_results = []
for node in nodes_of_interest:
    mask_ref = (df_early_candidates['node_extracted'] == node) | ((df_early_candidates['kpi_name'].str.contains(r'\bsystem\.', na=False)) & (df_early_candidates['cmdb_id'] == node))
    df_node = df_early_candidates[mask_ref].copy()
    if df_node.shape[0] == 0:
        node_kpi_results.append(pd.DataFrame(columns=['node','kpi_name','breach_count','earliest_breach_timestamp','max_breach_ratio']))
        continue
    # keep only breached points
    df_node_breach = df_node[df_node['is_anomaly']].copy()
    if df_node_breach.shape[0] == 0:
        node_kpi_results.append(pd.DataFrame(columns=['node','kpi_name','breach_count','earliest_breach_timestamp','max_breach_ratio']))
        continue
    agg = (
        df_node_breach
        .groupby('kpi_name', as_index=False)
        .agg(
            breach_count = ('kpi_name','count'),
            earliest_breach_timestamp = ('timestamp','min'),
            max_breach_ratio = ('ratio','max')
        )
    )
    agg['node'] = node
    # format ratio for sorting numeric; keep original for output formatting
    agg = agg[['node','kpi_name','breach_count','earliest_breach_timestamp','max_breach_ratio']]
    # sort by max_breach_ratio desc, limit top 10
    agg = agg.sort_values(by='max_breach_ratio', ascending=False).head(10).reset_index(drop=True)
    # format max_breach_ratio
    agg['max_breach_ratio'] = agg['max_breach_ratio'].map(fmt_ratio_val)
    node_kpi_results.append(agg)

# combine to a compact list of dicts per node
node_kpi_out = []
for dfn in node_kpi_results:
    if dfn.empty:
        # find node name from index if possible
        # We'll append an empty list representation for corresponding node in nodes_of_interest order
        node_kpi_out.append([])
    else:
        node = dfn.at[0,'node']
        records = []
        for _, r in dfn.iterrows():
            records.append({
                "node": node,
                "kpi_name": r['kpi_name'],
                "breach_count": int(r['breach_count']),
                "earliest_breach_timestamp": int(r['earliest_breach_timestamp']),
                "max_breach_ratio": r['max_breach_ratio']
            })
        node_kpi_out.append(records)

# Map nodes to their lists
node5_kpis = node_kpi_out[nodes_of_interest.index('node-5')]
node6_kpis = node_kpi_out[nodes_of_interest.index('node-6')]

# -----------------------------------------------------------------------------
# 2) Error-log window: adservice pods metrics and traces referencing adservice
# -----------------------------------------------------------------------------
ad_pods = ['adservice-0','adservice-1','adservice-2']

# Metrics: for each adservice pod, compute breaches in B window across all KPIs using precomputed global_P95
mask_ad_time = df_metric_with_p95['timestamp'].between(B_start, B_end) & df_metric_with_p95['cmdb_id'].isin(ad_pods)
df_ad_win = df_metric_with_p95[mask_ad_time].copy()
# drop rows without thresholds/values
df_ad_win = df_ad_win.dropna(subset=['value','global_P95']).copy()
df_ad_win['global_P95_f'] = df_ad_win['global_P95'].astype(float)
df_ad_win['ratio'] = np.where(
    df_ad_win['global_P95_f'] == 0,
    np.where(df_ad_win['value'] > 0, np.inf, 0.0),
    df_ad_win['value'] / df_ad_win['global_P95_f']
)
df_ad_win['is_anomaly'] = df_ad_win['value'] > df_ad_win['global_P95_f']

if df_ad_win.shape[0] == 0:
    ad_metric_summary = []
else:
    ad_agg = (
        df_ad_win[df_ad_win['is_anomaly']]
        .groupby('cmdb_id', as_index=False)
        .agg(
            breach_count = ('is_anomaly','sum'),
            earliest_breach_timestamp = ('timestamp','min'),
            max_breach_ratio = ('ratio','max')
        )
    )
    # ensure all pods are present even if zero
    rows = []
    for pod in ad_pods:
        if pod in ad_agg['cmdb_id'].values:
            r = ad_agg[ad_agg['cmdb_id']==pod].iloc[0]
            rows.append({
                "cmdb_id": pod,
                "breach_count": int(r['breach_count']),
                "earliest_breach_timestamp": int(r['earliest_breach_timestamp']),
                "max_breach_ratio": fmt_ratio_val(r['max_breach_ratio'])
            })
        else:
            rows.append({
                "cmdb_id": pod,
                "breach_count": 0,
                "earliest_breach_timestamp": None,
                "max_breach_ratio": None
            })
    ad_metric_summary = rows

# Traces: restrict trace.csv to window B, merge global_p95_trace, compute anomalies, filter trace_name containing 'adservice'
df_trace_win = df_trace[df_trace['timestamp'].between(B_start, B_end)].copy()
df_trace_win = df_trace_win.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')
df_trace_win = df_trace_win.dropna(subset=['value','global_P95']).copy()
df_trace_win['global_P95_f'] = df_trace_win['global_P95'].astype(float)
df_trace_win['ratio'] = np.where(
    df_trace_win['global_P95_f'] == 0,
    np.where(df_trace_win['value'] > 0, np.inf, 0.0),
    df_trace_win['value'] / df_trace_win['global_P95_f']
)
df_trace_win['is_anomaly'] = df_trace_win['value'] > df_trace_win['global_P95_f']

# filter trace_name referencing adservice (to_adservice or from_adservice or containing 'adservice')
mask_ad_trace = df_trace_win['trace_name'].str.contains(r'adservice', na=False)
df_trace_ad = df_trace_win[mask_ad_trace & df_trace_win['is_anomaly']].copy()

if df_trace_ad.shape[0] == 0:
    ad_trace_summary = []
else:
    trace_grp = (
        df_trace_ad
        .groupby('trace_name', as_index=False)
        .agg(
            breach_count = ('is_anomaly','sum'),
            sample_cmdb_id = ('cmdb_id', lambda s: s.iloc[0]),
            earliest_anomaly_timestamp_in_window = ('timestamp','min'),
            max_breach_ratio = ('ratio','max')
        )
    )
    # sort by breach_count desc then max_breach_ratio desc, limit 10
    trace_grp = trace_grp.sort_values(by=['breach_count','max_breach_ratio'], ascending=[False, False]).head(10).reset_index(drop=True)
    trace_out = []
    for _, r in trace_grp.iterrows():
        trace_out.append({
            "trace_name": r['trace_name'],
            "sample_cmdb_id": r['sample_cmdb_id'],
            "breach_count": int(r['breach_count']),
            "earliest_anomaly_timestamp_in_window": int(r['earliest_anomaly_timestamp_in_window']),
            "max_breach_ratio": fmt_ratio_val(r['max_breach_ratio'])
        })
    ad_trace_summary = trace_out

# Final compact result structure
final_result = {
    "early_window_node_kpis": {
        "node-6_top_kpis": node6_kpis,
        "node-5_top_kpis": node5_kpis
    },
    "adservice_error_window": {
        "metrics_summary": ad_metric_summary,
        "trace_summary": ad_trace_summary
    }
}

final_result

```
Out[9]:
```
Summary of the two compact checks:

1) Early window (1647736200–1647736320) — node-6 vs node-5 top breached node-specific KPIs
- node-6 (top breached KPIs by max breach ratio):
  - container.node-6.container_network_receive_MB.eth0 — breach_count=8, earliest=1647736200, max_breach_ratio≈17.192012
  - container.node-6.container_cpu_cfs_throttled_seconds — breach_count=1, earliest=1647736200, max_breach_ratio≈5.552953
  - container.node-6.container_cpu_cfs_throttled_periods — breach_count=1, earliest=1647736200, max_breach_ratio=2.0
  - container.node-6.container_cpu_system_seconds — breach_count=2, earliest=1647736200, max_breach_ratio=1.5
  - system.io.w_s and other IO/fs metrics and additional network KPIs also breached with smaller ratios.
  - Interpretation: node-6 extreme ratios are driven primarily by network-related KPIs (e.g., container_network_receive_MB.eth0) and notable CPU throttling metrics.

- node-5 (top breached KPIs by max breach ratio):
  - container.node-5.container_network_receive_MB.eth0 — breach_count=2, earliest=1647736200, max_breach_ratio≈16.91028
  - system.io.r_await — breach_count=1, earliest=1647736200, max_breach_ratio≈2.439024
  - system.io.await, system.io.svctm, system.io.rkb_s — single breaches with moderate ratios
  - container.node-5.container_network_transmit_packets.eth0 and other network/packet KPIs also appear.
  - Interpretation: node-5 also shows very large network-related breaches (receive MB) and some I/O symptoms.

Bottom line for check #1: both node-6 and node-5 show extremely large breach ratios in the early window, and the highest-ratio KPIs are network throughput metrics (container_network_receive_MB.eth0 and related network KPIs). This confirms network-related node KPIs are the primary source of the extreme ratios for these nodes in the early window.

2) Error-log window (1647737332–1647737513) — adservice pods (metrics) and adservice-related traces
- Metrics (per adservice pod):
  - adservice-0 — breach_count=22, earliest_breach_timestamp=1647737340, max_breach_ratio=inf
  - adservice-1 — breach_count=18, earliest_breach_timestamp=1647737340, max_breach_ratio=inf
  - adservice-2 — breach_count=30, earliest_breach_timestamp=1647737340, max_breach_ratio=inf
  - Interpretation: All three adservice pods had many metric breaches in the error-log window. max_breach_ratio='inf' indicates those KPIs had global_P95 == 0 while observed values > 0 during the window (i.e., a jump from zero baseline).

- Traces (adservice-related, top entries):
  - trace.to_adservice-2.duration_p95 — sample_cmdb_id=frontend-0, breach_count=1, earliest=1647737400, max_breach_ratio≈1.454545
  - trace.to_adservice-2.duration_mean — sample_cmdb_id=frontend-0, breach_count=1, earliest=1647737400, max_breach_ratio≈1.350482
  - trace.to_adservice-0.duration_mean — sample_cmdb_id=frontend-0, breach_count=1, earliest=1647737400, max_breach_ratio≈1.089109
  - trace.to_adservice-0.duration_p95 — sample_cmdb_id=frontend-0, breach_count=1, earliest=1647737400, max_breach_ratio≈1.078509
  - Interpretation: A few trace metrics referencing adservice (mostly to_adservice from frontend-0) breached their P95 in the error-log window with moderate ratios (~1.08–1.45).

Overall conclusion:
- Early window anomalies on node-5 and node-6 are dominated by network throughput KPIs (very large breach ratios).
- During the error-log window, adservice pods show many metric breaches (often with global_P95 == 0 → observed > 0, recorded as infinite ratio), and traces indicate increased latency to adservice (moderate breach ratios) originating from frontend-0.

The original code execution output of IPython Kernel is also provided below for reference:

{'early_window_node_kpis': {'node-6_top_kpis': [{'node': 'node-6', 'kpi_name': 'container.node-6.container_network_receive_MB.eth0', 'breach_count': 8, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 17.192012}, {'node': 'node-6', 'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'breach_count': 1, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 5.552953}, {'node': 'node-6', 'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'breach_count': 1, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 2.0}, {'node': 'node-6', 'kpi_name': 'container.node-6.container_cpu_system_seconds', 'breach_count': 2, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.5}, {'node': 'node-6', 'kpi_name': 'system.io.w_s', 'breach_count': 1, 'earliest_breach_timestamp': 1647736320, 'max_breach_ratio': 1.480392}, {'node': 'node-6', 'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'breach_count': 5, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.430928}, {'node': 'node-6', 'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'breach_count': 8, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.210127}, {'node': 'node-6', 'kpi_name': 'system.io.w_await', 'breach_count': 1, 'earliest_breach_timestamp': 1647736260, 'max_breach_ratio': 1.169014}, {'node': 'node-6', 'kpi_name': 'system.io.await', 'breach_count': 1, 'earliest_breach_timestamp': 1647736260, 'max_breach_ratio': 1.169014}, {'node': 'node-6', 'kpi_name': 'container.node-6.container_network_transmit_MB.eth0', 'breach_count': 3, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.146071}], 'node-5_top_kpis': [{'node': 'node-5', 'kpi_name': 'container.node-5.container_network_receive_MB.eth0', 'breach_count': 2, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 16.91028}, {'node': 'node-5', 'kpi_name': 'system.io.r_await', 'breach_count': 1, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 2.439024}, {'node': 'node-5', 'kpi_name': 'system.io.await', 'breach_count': 1, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.449275}, {'node': 'node-5', 'kpi_name': 'system.io.svctm', 'breach_count': 1, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.416667}, {'node': 'node-5', 'kpi_name': 'system.io.rkb_s', 'breach_count': 1, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.294118}, {'node': 'node-5', 'kpi_name': 'container.node-5.container_network_transmit_packets.eth0', 'breach_count': 2, 'earliest_breach_timestamp': 1647736260, 'max_breach_ratio': 1.265421}, {'node': 'node-5', 'kpi_name': 'container.node-5.container_cpu_user_seconds', 'breach_count': 1, 'earliest_breach_timestamp': 1647736200, 'max_breach_ratio': 1.166667}, {'node': 'node-5', 'kpi_name': 'container.node-5.container_network_receive_packets.eth0', 'breach_count': 2, 'earliest_breach_timestamp': 1647736260, 'max_breach_ratio': 1.156334}, {'node': 'node-5', 'kpi_name': 'system.cpu.pct_usage', 'breach_count': 1, 'earliest_breach_timestamp': 1647736320, 'max_breach_ratio': 1.059418}, {'node': 'node-5', 'kpi_name': 'system.net.udp.in_datagrams', 'breach_count': 1, 'earliest_breach_timestamp': 1647736260, 'max_breach_ratio': 1.045683}]}, 'adservice_error_window': {'metrics_summary': [{'cmdb_id': 'adservice-0', 'breach_count': 22, 'earliest_breach_timestamp': 1647737340, 'max_breach_ratio': 'inf'}, {'cmdb_id': 'adservice-1', 'breach_count': 18, 'earliest_breach_timestamp': 1647737340, 'max_breach_ratio': 'inf'}, {'cmdb_id': 'adservice-2', 'breach_count': 30, 'earliest_breach_timestamp': 1647737340, 'max_breach_ratio': 'inf'}], 'trace_summary': [{'trace_name': 'trace.to_adservice-2.duration_p95', 'sample_cmdb_id': 'frontend-0', 'breach_count': 1, 'earliest_anomaly_timestamp_in_window': 1647737400, 'max_breach_ratio': 1.454545}, {'trace_name': 'trace.to_adservice-2.duration_mean', 'sample_cmdb_id': 'frontend-0', 'breach_count': 1, 'earliest_anomaly_timestamp_in_window': 1647737400, 'max_breach_ratio': 1.350482}, {'trace_name': 'trace.to_adservice-0.duration_mean', 'sample_cmdb_id': 'frontend-0', 'breach_count': 1, 'earliest_anomaly_timestamp_in_window': 1647737400, 'max_breach_ratio': 1.089109}, {'trace_name': 'trace.to_adservice-0.duration_p95', 'sample_cmdb_id': 'frontend-0', 'breach_count': 1, 'earliest_anomaly_timestamp_in_window': 1647737400, 'max_breach_ratio': 1.078509}]}}```
```